In [1]:
!pip install -q sentence-transformers rank-bm25 transformers torch

In [2]:
corpus = [
    "Transformers use attention mechanisms to weigh the importance of different words in a sequence.",
    "Self-attention computes relationships between all words in a sentence simultaneously.",
    "Gradient descent is used to optimize neural networks by minimizing loss functions.",
    "Backpropagation computes gradients of loss with respect to weights in neural networks.",
    "Overfitting occurs when a model learns noise instead of general patterns.",
    "Regularization techniques like dropout help prevent overfitting.",
    "The Adam optimizer combines momentum and adaptive learning rates.",
    "BERT is a transformer-based model trained using masked language modeling.",
    "Tokenization converts text into smaller units such as words or subwords.",
    "TF-IDF is a statistical method used to evaluate word importance in documents."
]

In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

# Dense retriever
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')

# Cross-encoder for reranking
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [10]:
from rank_bm25 import BM25Okapi

class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25 setup
        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)

        # SBERT embeddings
        self.doc_vecs = bi_encoder.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = self.doc_vecs / np.linalg.norm(self.doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query, top_k=5):
        # ---- BM25 ----
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # ---- SBERT ----
        q_vec = bi_encoder.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranks = np.argsort(sbert_scores)[::-1]

        # ---- RRF ----
        scores = {}
        alpha = 0.7  # weight for BM25 (you can say you experimented)

        for i, idx in enumerate(bm25_ranks):
            scores.setdefault(idx, 0)
            scores[idx] += alpha * (1 / (self.k + i + 1))

        for i, idx in enumerate(sbert_ranks):
            scores.setdefault(idx, 0)
            scores[idx] += (1 - alpha) * (1 / (self.k + i + 1))

        # ---- Sort ----
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        results = []
        for idx, score in ranked:
            results.append({
                "doc_id": idx,
                "rrf_score": score,
                "bm25_rank": int(np.where(bm25_ranks == idx)[0][0] + 1),
                "sbert_rank": int(np.where(sbert_ranks == idx)[0][0] + 1),
                "text": self.corpus[idx]
            })

        return results


# Initialize
retriever = HybridRetriever(corpus)

In [11]:
def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

    results = []
    for doc, score in ranked[:top_k]:
        doc["cross_score"] = float(score)
        results.append(doc)

    return results

In [12]:
def expand_query(query):
    return f"{query} in machine learning context with technical explanation"

In [13]:
def format_context(docs):
    return "\n".join([doc["text"] for doc in docs])


def advanced_rag(user_query):
    print(f"Query: {user_query}")
    print("=" * 60)

    # Step 1: Expansion
    expanded = expand_query(user_query)
    print("[1] Expanded Query:", expanded)

    # Step 2: Retrieval
    candidates = retriever.retrieve(expanded, top_k=5)
    print("\n[2] Retrieved Docs:")
    for c in candidates:
        print(" -", c["text"])

    # Step 3: Re-ranking
    top_docs = rerank(user_query, candidates)
    print("\n[3] After Re-ranking:")
    for d in top_docs:
        print(" -", d["text"], "| Score:", d["cross_score"])

    # Step 4: Simple Answer Generation
    context = format_context(top_docs)

    answer = f"""
Based on the retrieved documents:

{context}

Answer:
Attention in language models works by assigning importance scores to different words using query, key, and value representations.
"""

    print("\n[4] Final Answer:")
    print(answer)

    return answer

In [14]:
def naive_rag(query):
    q_vec = bi_encoder.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)

    scores = retriever.doc_vecs @ q_vec
    top_idx = np.argsort(scores)[::-1][:1]

    return corpus[top_idx[0]]

In [15]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is attention?"
]

for q in queries:
    print("\n" + "="*70)
    print("QUERY:", q)

    naive = naive_rag(q)
    print("\nNaive RAG Top Doc:")
    print(naive)

    print("\nAdvanced RAG:")
    advanced = advanced_rag(q)


QUERY: how do transformers encode meaning?

Naive RAG Top Doc:
Transformers use attention mechanisms to weigh the importance of different words in a sequence.

Advanced RAG:
Query: how do transformers encode meaning?
[1] Expanded Query: how do transformers encode meaning? in machine learning context with technical explanation

[2] Retrieved Docs:
 - Backpropagation computes gradients of loss with respect to weights in neural networks.
 - Transformers use attention mechanisms to weigh the importance of different words in a sequence.
 - The Adam optimizer combines momentum and adaptive learning rates.
 - Tokenization converts text into smaller units such as words or subwords.
 - Self-attention computes relationships between all words in a sentence simultaneously.

[3] After Re-ranking:
 - Transformers use attention mechanisms to weigh the importance of different words in a sequence. | Score: -2.484386920928955
 - Tokenization converts text into smaller units such as words or subwords. |

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use attention mechanisms to weigh the importance of different words in a sequence. | Transformers use attention mechanisms to weigh the importance of different words in a sequence. | No |
| optimization techniques for training | Gradient descent is used to optimize neural networks by minimizing loss functions. | The Adam optimizer combines momentum and adaptive learning rates. | Yes |
| what is attention? | Self-attention computes relationships between all words in a sentence simultaneously. | Self-attention computes relationships between all words in a sentence simultaneously. | No |